In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from tabpfn import TabPFNClassifier
import warnings
warnings.filterwarnings("ignore")

CSV_PATH = r"/home/protein/Desktop/ESM_clf/comp_feat/merged_comp_feat.csv"

# Load data
df = pd.read_csv(CSV_PATH)
assert "label" in df.columns, "Column 'label' not found!"

X = df.drop(columns=["label"])
y = df["label"].values

In [2]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

accs, lls, precs, recs, f1s = [], [], [], [], []
cms = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\nFold {fold}")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    model = TabPFNClassifier(device="cuda")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    y_prob = model.predict_proba(X_val)

    accs.append(accuracy_score(y_val, y_pred))
    lls.append(log_loss(y_val, y_prob))
    precs.append(precision_score(y_val, y_pred, zero_division=0))
    recs.append(recall_score(y_val, y_pred))
    f1s.append(f1_score(y_val, y_pred))
    cms.append(confusion_matrix(y_val, y_pred))

    print(classification_report(y_val, y_pred))


Fold 1
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      3093
           1       0.87      0.79      0.83      1433

    accuracy                           0.90      4526
   macro avg       0.89      0.87      0.88      4526
weighted avg       0.90      0.90      0.89      4526


Fold 2
              precision    recall  f1-score   support

           0       0.92      0.94      0.93      3093
           1       0.87      0.82      0.85      1433

    accuracy                           0.91      4526
   macro avg       0.90      0.88      0.89      4526
weighted avg       0.90      0.91      0.90      4526


Fold 3
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      3092
           1       0.87      0.80      0.84      1433

    accuracy                           0.90      4525
   macro avg       0.89      0.87      0.88      4525
weighted avg       0.90      0.90      0.90      4

In [3]:
print(f"Accuracy  : {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Log Loss  : {np.mean(lls):.4f} ± {np.std(lls):.4f}")
print(f"Precision : {np.mean(precs):.4f} ± {np.std(precs):.4f}")
print(f"Recall    : {np.mean(recs):.4f} ± {np.std(recs):.4f}")
print(f"F1 Score  : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

print("\nMean Confusion Matrix:")
print(np.mean(cms, axis=0).astype(int))

Accuracy  : 0.8990 ± 0.0070
Log Loss  : 0.2401 ± 0.0139
Precision : 0.8706 ± 0.0068
Recall    : 0.7998 ± 0.0186
F1 Score  : 0.8336 ± 0.0128

Mean Confusion Matrix:
[[2922  170]
 [ 286 1145]]


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from tabpfn import TabPFNClassifier
import warnings
warnings.filterwarnings("ignore")

CSV_PATH = r"/home/protein/Desktop/ESM_clf/comp_feat/merged_comp_feat.csv"

df = pd.read_csv(CSV_PATH)
assert "label" in df.columns, "Column 'label' not found!"

X = df.drop(columns=["label"])
y = df["label"]




In [2]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


In [3]:
model = TabPFNClassifier(device="cuda")

model.fit(X_train, y_train)

,n_estimators,8
,categorical_features_indices,None
,softmax_temperature,0.9
,balance_probabilities,False
,average_before_softmax,False
,model_path,'auto'
,device,'cuda'
,ignore_pretraining_limits,False
,inference_precision,'auto'
,fit_mode,'fit_preprocessors'
,memory_saving_mode,'auto'


In [4]:
y_pred = model.predict(X_temp)
y_prob = model.predict_proba(X_temp)

acc = accuracy_score(y_temp, y_pred)
ll = log_loss(y_temp, y_prob)
prec = precision_score(y_temp, y_pred)
rec = recall_score(y_temp, y_pred)
f1 = f1_score(y_temp, y_pred)
cm = confusion_matrix(y_temp, y_pred)

print("--------------------------------")
print(f"Accuracy     : {acc:.4f}")
print(f"Log Loss     : {ll:.4f}")
print(f"Precision    : {prec:.4f}")
print(f"Recall       : {rec:.4f}")
print(f"F1 Score     : {f1:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_temp, y_pred))

--------------------------------
Accuracy     : 0.9107
Log Loss     : 0.2163
Precision    : 0.8866
Recall       : 0.8234
F1 Score     : 0.8538

Confusion Matrix:
[[2942  151]
 [ 253 1180]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.95      0.94      3093
           1       0.89      0.82      0.85      1433

    accuracy                           0.91      4526
   macro avg       0.90      0.89      0.89      4526
weighted avg       0.91      0.91      0.91      4526



In [5]:
from tabpfn.model_loading import (
    load_fitted_tabpfn_model,
    save_fitted_tabpfn_model,
)
save_fitted_tabpfn_model(model, "Tabpfn_comp_feat.tabpfn_fit")